FAIS Vector Stores - Facebook AI Similarity Search


In [3]:
#FAIS Vector Stores - Facebook AI Similarity Search

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings 
from langchain_text_splitters import CharacterTextSplitter

/Users/satyakibasu/Documents/Satyaki/python_code/gen-ai/my-langchain/p310env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=30)
docs = text_splitter.split_documents(documents)

In [ ]:
docs

In [8]:
embeddings = OllamaEmbeddings(model = 'mistral') #'mistral' has been downloaded
db = FAISS.from_documents(docs, embedding=embeddings)


In [ ]:
## Query the vector database

query = "What does the speaker belive that United States should enter the war?"
results = db.similarity_search(query)
results[0].page_content

# Asynch query
results = await db.asimilarity_search(query)
results[0].page_content

In [ ]:
## Use as Retriever

retriever = db.as_retriever() # Converts the vector stores as Retriever Class which can be used in other LangChain methods
results = retriever.invoke(query)
results

In [ ]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

In [ ]:
# Convert the query into vector and do a similarity search by vectors

query_vector = embeddings.embed_query(query)
query_vector

results = db.similarity_search_by_vector(query_vector)
results

In [ ]:
# Save the Vector DB on local
db.save_local("faiss_index")

In [6]:
# Load the db from local

new_db = FAISS.load_local("faiss_index", embeddings=embeddings,allow_dangerous_deserialization=True) # Ollama Embeddings

In [ ]:
results = new_db.similarity_search(query)
results

Using Chroma DB

In [ ]:
#from langchain_community.vectorstores import Chroma # this is one way
from langchain_chroma import Chroma # This is another recently used way
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter2 = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs2 = text_splitter2.split_documents(documents)

In [ ]:
chroma_db = Chroma.from_documents(documents=docs2,embedding=embeddings) 
chroma_db

In [ ]:
query = "What does the speaker belive that United States should enter the war?"
results = chroma_db.similarity_search(query)
results[0].page_content

In [ ]:
# Saving database to disk
chroma_db = Chroma.from_documents(documents=docs2,embedding=embeddings,persist_directory="./chroma_db") 

In [ ]:
# Load from disk
db2 = Chroma(persist_directory="./chroma_db",embedding_function=embeddings)

In [ ]:
results = db2.similarity_search(query)
results

In [ ]:
# Retrievers using multiple techniques
 
from typing import List
#from langchain_core.documents import Documents
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(db.similarity_search).bind(k=1)
retriever.batch(['cats','dog'])


# 2nd option
retriever2 = db.as_retriever(
    search_type="similarity",search_kwargs={"k":1}
)
retriever2.batch(['pride','united'])





[[Document(id='7db42f7f-d2e1-43e9-9839-46c4d58ec298', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.')],
 [Document(id='836ec12f-bb93-480a-9a94-b5651dcffbff', metadata={'source': 'speech.txt'}, page_content='

In [14]:
## RAG
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message = """
Answer this question using the provided context only.

{question}

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([("human", message)])

rag_chain={"context":retriever,"question":RunnablePassthrough()}|prompt|llm

response=rag_chain.invoke("tell me about dogs")
print(response.content)

NameError: name 'llm' is not defined